# ProboSed — 01: JCORES VCD Mining and MTD Catalog

**IODP Expedition 405 — Site C0019J, Japan Trench**

This notebook extracts stability scores from JCORES-format VCD PDFs and
produces the MTD boundary catalog used by all downstream notebooks.

**What this notebook does:**
1. Builds a depth backbone from the core summary Excel file
2. Extracts stability scores from the JCORES VCD PDF using standardized IODP disturbance terminology
3. Identifies MTD boundaries as contiguous intervals where stability score <= 1
4. Saves a stability log CSV and MTD catalog CSV for use in notebooks 02-04

**Why JCORES only (current version):**
JCORES PDFs are machine-readable structured digital documents generated by
the shipboard database system. Text extraction is reliable and requires no
image processing or handwriting recognition. The interactive VCDLabeler
pathway (for scanned handwritten VCD forms) is reserved for future work
pending development of a sedimentologist-specific handwriting recognition
model trained on IODP expedition handwriting samples.

**Output files:**
- `C0019J_VCD_stability_log.csv` — stability score per depth interval
- `C0019J_MTD_catalog.csv` — MTD boundaries for use in notebooks 03 and 04
- `C0019J_stability_profile.png` — visual QC of the extracted profile

In [ ]:
# Cell 1: Install dependencies
# pymupdf (fitz) is required for PDF text extraction.
# openpyxl is required for reading the core summary Excel file.
!pip install pymupdf openpyxl pandas numpy matplotlib --quiet

In [ ]:
# Cell 2: Mount Google Drive and clone ProboSed
# The ProboSed .py modules live in the repository py/ folder.
# After cloning, they are added to the Python path for import.
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

# Update repo_url to the actual repository URL before running
repo_url  = 'https://github.com/yourusername/ProboSed.git'
repo_path = '/content/ProboSed'

result = subprocess.run(['git', 'clone', repo_url, repo_path],
                        capture_output=True, text=True)
if result.returncode != 0:
    # repository already present — pull latest changes instead
    subprocess.run(['git', '-C', repo_path, 'pull'], capture_output=True)

# add the py/ subfolder to the Python path
sys.path.insert(0, repo_path)
print('ProboSed modules available')

In [ ]:
# Cell 3: Configure paths
# Update DATA_DIR to match the Google Drive folder containing the data files.
# All output files are written to OUTPUT_DIR.
import os

DATA_DIR   = '/content/drive/MyDrive/iodp/X405/Data & Data Tracking'
OUTPUT_DIR = '/content/drive/MyDrive/iodp/X405/ProboSed_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# JCORES VCD PDF — machine-readable digital text, not scanned handwriting
VCD_PDF = os.path.join(DATA_DIR, 'vcds/405-C0019J_VCD.pdf')

# Core summary Excel file — provides authoritative CSF-A depths per core
SUMMARY_XLSX = os.path.join(DATA_DIR, 'Summary_C0019J.xlsx')

# Output paths
STABILITY_CSV = os.path.join(OUTPUT_DIR, 'C0019J_VCD_stability_log.csv')
MTD_CSV       = os.path.join(OUTPUT_DIR, 'C0019J_MTD_catalog.csv')
PROFILE_PNG   = os.path.join(OUTPUT_DIR, 'C0019J_stability_profile.png')

# Verify input files exist before proceeding
for path, label in [(VCD_PDF, 'VCD PDF'), (SUMMARY_XLSX, 'Core summary Excel')]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND — check path'
    print(f'{label}: {status}')
    print(f'  {path}')

## Step 1 — Build depth backbone

The backbone assigns a CSF-A depth to every core section based on the
authoritative top and bottom depths from the core summary Excel file.
Section depths are interpolated at 1.5 m intervals (standard IODP section length).

This is the depth reference used for all subsequent analyses.

In [ ]:
# Cell 4: Build depth backbone from core summary
from core_ml.labeler import JCORESMiner

miner    = JCORESMiner(VCD_PDF, SUMMARY_XLSX)
backbone = miner.build_backbone()

print(f'\nBackbone sample (first 10 rows):')
backbone.head(10)

## Step 2 — Extract stability scores from the JCORES VCD PDF

The miner scans every page of the PDF and:
1. Identifies the core and section from JCORES-format identifiers
2. Scores stability from the JCORES standardized disturbance vocabulary using a
   lowest-score-wins rule — if multiple terms appear on a page, the most severe governs
3. Matches the (Core, Section) pair to the backbone to assign a CSF-A depth

Pages that cannot be matched to a backbone entry are still recorded with
Depth_m = NaN so they can be inspected for extraction quality.

In [ ]:
# Cell 5: Extract stability scores from the VCD PDF
vcd_df = miner.extract(backbone)

# save the full stability log
vcd_df.to_csv(STABILITY_CSV, index=False)
print(f'Saved: {STABILITY_CSV}')

# show the score distribution
print(f'\nStability score distribution:')
print(vcd_df['Stability'].value_counts().sort_index())
print(f'\nDepth-matched pages: {vcd_df["Depth_m"].notna().sum()} / {len(vcd_df)}')
vcd_df.head(10)

## Step 3 — Build the MTD boundary catalog

MTD intervals are defined as contiguous depth sequences where stability <= 1,
meaning scaly fabric (1) or slurried / chaotic material (0).

This threshold captures intervals that have undergone significant fabric
destruction consistent with mass transport, while excluding minor disturbance
(coherent block, score 2) which may be drilling-related rather than tectonic.

The catalog is the key pipeline output — it is used by:
- **Notebook 04** to shade MTD intervals on geochemistry profiles
- **Notebook 02/03** via `threshold_from_vcd()` to set per-depth failure thresholds

In [ ]:
# Cell 6: Build MTD boundary catalog
mtd_catalog = JCORESMiner.score_to_mtd_catalog(vcd_df, stability_threshold=1)

# save the catalog
mtd_catalog.to_csv(MTD_CSV, index=False)
print(f'MTD intervals identified: {len(mtd_catalog)}')
print(f'Saved: {MTD_CSV}')
print()
mtd_catalog

## Step 4 — Visual QC of the stability profile

Quick check that the extraction looks physically reasonable:
- Intact bedding (score 3) should dominate the upper prism
- Low-score intervals (0-1) should cluster near thrust fault boundaries
- The plate boundary fault zone (~820 mbsf) should show score 0

In [ ]:
# Cell 7: Plot stability profile with MTD intervals shaded
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# drop unmatched pages for plotting
plot_df = vcd_df.dropna(subset=['Depth_m', 'Stability']).sort_values('Depth_m')

fig, ax = plt.subplots(figsize=(6, 16))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# stability profile as a step function
ax.step(plot_df['Stability'], plot_df['Depth_m'],
        where='post', color='#2c3e50', lw=1.5, zorder=3)

# shade MTD intervals from the catalog
for _, row in mtd_catalog.iterrows():
    ax.axhspan(row['top_m'], row['bottom_m'],
               alpha=0.15, color='orange', zorder=1)

# mark the plate boundary fault zone
ax.axhline(820, color='red', ls='--', lw=1.5, alpha=0.8,
           label='Plate boundary fault (~820 mbsf)')

ax.invert_yaxis()   # depth increases downward
ax.set_xlim(-0.5, 3.5)
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Slurried\n(0)', 'Scaly\n(1)', 'Coherent\n(2)', 'Intact\n(3)'],
                   fontsize=10)
ax.set_ylabel('Depth (m CSF-A)', fontsize=12)
ax.set_title('C0019J Stability Profile\n(JCORESMiner extraction)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)

# add MTD count annotation
ax.text(0.02, 0.02, f'MTD intervals: {len(mtd_catalog)}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='orange', alpha=0.2))

plt.tight_layout()
plt.savefig(PROFILE_PNG, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {PROFILE_PNG}')

In [ ]:
# Cell 8: Summary of key MTD intervals
# Print a readable summary of identified MTD boundaries.
# These depths should be checked against published structural interpretations
# (Nakamura et al. 2020; Conin et al. in review) for cross-validation.
print('MTD boundary summary:')
print(f'  Total intervals:       {len(mtd_catalog)}')
if len(mtd_catalog) > 0:
    print(f'  Shallowest top:        {mtd_catalog["top_m"].min():.1f} m CSF-A')
    print(f'  Deepest base:          {mtd_catalog["bottom_m"].max():.1f} m CSF-A')
    print(f'  Mean thickness:        {mtd_catalog["thickness_m"].mean():.1f} m')
    print(f'  Total MTD thickness:   {mtd_catalog["thickness_m"].sum():.1f} m')
    print()
    print('Individual intervals:')
    for _, row in mtd_catalog.iterrows():
        print(f'  {row["mtd_id"]}: {row["top_m"]:.1f} - {row["bottom_m"]:.1f} m  '
              f'({row["thickness_m"]:.1f} m thick, '
              f'mean stability = {row["mean_stability"]:.1f})')

## Extraction QC — pages without depth matches

Pages that could not be matched to a backbone depth (Depth_m = NaN) are
typically cover pages, figure pages, or pages where the core/section
identifier format did not match the regex pattern.

Inspect these below. If important depth intervals are missing, the
JCORES_LEXICON or CORE_PATTERN regex in `labeler.py` may need adjustment.

In [ ]:
# Cell 9: Inspect unmatched pages
unmatched = vcd_df[vcd_df['Depth_m'].isna()][['PDF_Page', 'Core', 'Section', 'Disturbance', 'Snippet']]
print(f'Unmatched pages: {len(unmatched)} / {len(vcd_df)} total')
if len(unmatched) > 0:
    print('\nFirst 20 unmatched pages:')
    print(unmatched.head(20).to_string(index=False))

---

## Future work — VCDLabeler for handwritten VCD forms

The `VCDLabeler` class in `labeler.py` provides an interactive image-by-image
labeling interface for core section photographs. It is designed for application
to scanned handwritten VCD forms from expeditions that predate the JCORES
digital system.

**Current status:** The interface is implemented but not yet deployable for
production use because it requires a library of sedimentologist handwriting
samples to ensure consistent classification across observers and expeditions.

**To activate this pathway in a future version:**
1. Collect handwriting samples from the target sedimentologists across the
   target expeditions (386, 343, and earlier)
2. Train or fine-tune a handwriting recognition model on those samples
3. Integrate model predictions as default dropdown suggestions in the labeler
4. Validate against the JCORES ground truth established by this notebook

The code is in place — only the handwriting model is missing.